In [1]:
import pandas as pd
import requests
import json
from utlis import get_api_entry_by_llm, get_data, merge_by_qid
from question_classify import classify

In [2]:
model_embedding = get_api_entry_by_llm("LLM embedding")


def get_embedding(text, model):
    header = {
        "Content-Type": "application/json",
        "Authorization": model["authorization"],
        "Token-id": model["tokenId"],
        "Token-key": model["tokenKey"],
    }    
    
    json_data = {
        'model': 'vnptai_hackathon_embedding', 
        'input': text, 
        'encoding_format': 'float'
    }
    response = requests.post("https://api.idg.vnpt.vn/data-service/vnptai-hackathon-embedding", headers=header, json=json_data)
    result = response.json()
    embedding = result["data"][0]["embedding"]
    return embedding

In [3]:
import numpy as np

def get_embedding_dim():
    v = get_embedding("test")
    return len(v)

def get_embeddings_batch(texts):
    # nếu model support batch thì dùng trực tiếp
    # ở đây giả sử chưa có -> loop
    return np.vstack([get_embedding(t) for t in texts])


In [4]:
def count_tokens(text: str) -> int:
    # approx: 1 token ≈ 4 chars (tiếng Anh). Với tiếng Việt thì vẫn tạm ổn để limit.
    return max(1, len(text) // 4)

In [5]:
def chunk_text(text, max_tokens=512, overlap_tokens=64):
    tokens = []
    # simple split theo câu / dòng
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current = []
    current_tokens = 0

    for sent in sentences:
        t = count_tokens(sent)
        if current_tokens + t > max_tokens and current:
            chunks.append(" ".join(current))
            # overlap
            while current and current_tokens > overlap_tokens:
                removed = current.pop(0)
                current_tokens -= count_tokens(removed)
        current.append(sent)
        current_tokens += t

    if current:
        chunks.append(" ".join(current))
    return chunks


In [6]:
import json
from pathlib import Path

def iter_wiki_records(wiki_dir="wiki"):
    wiki_path = Path(wiki_dir)
    for f in wiki_path.glob("*.jsonl"):
        with f.open("r", encoding="utf-8") as fh:
            for line in fh:
                if not line.strip():
                    continue
                obj = json.loads(line)
                yield obj  # {id, url, title, text}


In [7]:
def iter_chunks_with_meta(wiki_dir="wiki", max_tokens=512, overlap_tokens=64):
    chunk_id = 0
    for rec in iter_wiki_records(wiki_dir):
        doc_id = rec["id"]
        url = rec.get("url")
        title = rec.get("title")
        text = rec.get("text", "")
        chunks = chunk_text(text, max_tokens=max_tokens, overlap_tokens=overlap_tokens)
        for i, ch in enumerate(chunks):
            meta = {
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "url": url,
                "title": title,
                "chunk_index": i,
            }
            yield ch, meta
            chunk_id += 1


In [8]:
import faiss
import numpy as np
import json

def build_faiss_index(
    wiki_dir="wiki",
    index_out="faiss.index",
    meta_out="metadata.jsonl",
    max_tokens=512,
    overlap_tokens=64,
    batch_size=64,
    use_ivf=True,
    nlist=4096,
):
    dim = get_embedding_dim()
    print("Embedding dim:", dim)

    # Flat index (cosine) + L2 normalize
    quantizer = faiss.IndexFlatIP(dim)
    if use_ivf:
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    else:
        index = quantizer

    # 1. Thu sample để train IVF (nếu dùng)
    if use_ivf:
        print("Collecting training samples for IVF...")
        sample_vectors = []
        for i, (chunk, _) in enumerate(iter_chunks_with_meta(wiki_dir, max_tokens, overlap_tokens)):
            sample_vectors.append(get_embedding(chunk))
            if (i + 1) >= 20000:  # lấy 20k sample để train là đủ
                break
        sample_matrix = np.vstack(sample_vectors).astype("float32")
        # normalize
        faiss.normalize_L2(sample_matrix)
        print("Training IVF index...")
        index.train(sample_matrix)
        print("Training done.")

    # 2. Build index thật
    print("Building index...")
    metas = []
    batch_texts = []
    batch_meta = []

    for chunk, meta in iter_chunks_with_meta(wiki_dir, max_tokens, overlap_tokens):
        batch_texts.append(chunk)
        batch_meta.append(meta)

        if len(batch_texts) >= batch_size:
            embs = get_embeddings_batch(batch_texts).astype("float32")
            faiss.normalize_L2(embs)
            index.add(embs)
            metas.extend(batch_meta)
            batch_texts = []
            batch_meta = []

    # flush phần còn lại
    if batch_texts:
        embs = get_embeddings_batch(batch_texts).astype("float32")
        faiss.normalize_L2(embs)
        index.add(embs)
        metas.extend(batch_meta)

    print("Total vectors:", index.ntotal)

    # 3. Save index
    faiss.write_index(index, index_out)
    print("Index saved to", index_out)

    # 4. Save metadata (streaming JSONL)
    with open(meta_out, "w", encoding="utf-8") as f:
        for m in metas:
            f.write(json.dumps(m, ensure_ascii=False) + "\n")
    print("Metadata saved to", meta_out)


In [9]:
import faiss
import json

class WikiVectorDB:
    def __init__(self, index_path="faiss.index", meta_path="metadata.jsonl"):
        self.index = faiss.read_index(index_path)
        self.metadatas = []
        with open(meta_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.metadatas.append(json.loads(line))

    def search(self, query, k=5):
        q_emb = get_embedding(query).astype("float32")
        q_emb = np.expand_dims(q_emb, axis=0)
        faiss.normalize_L2(q_emb)
        scores, idxs = self.index.search(q_emb, k)
        idxs = idxs[0]
        scores = scores[0]
        results = []
        for rank, (i, s) in enumerate(zip(idxs, scores)):
            if i < 0:
                continue
            meta = self.metadatas[i]
            meta["score"] = float(s)
            results.append(meta)
        return results


In [ ]:
build_faiss_index()